# Leçon 08 - Modèle de conception Multi-Agent


## Configuration


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Pourquoi les systèmes multi-agents ?

Les tâches du monde réel, comme la planification de voyages, impliquent de nombreux types de compétences — logistique, connaissance locale, budgétisation, et plus encore. Un agent unique qui essaie de tout gérer devient rapidement ingérable.

Les systèmes multi-agents résolvent ce problème par la **spécialisation** : chaque agent se concentre sur un domaine d'expertise, produisant des résultats de meilleure qualité qu'un généraliste. Ils améliorent également la **scalabilité** — vous pouvez ajouter de nouveaux agents (par exemple, un spécialiste des vols, un critique de restaurants) sans réécrire le flux de travail existant. Les agents s'assemblent ensemble à travers un pipeline structuré, passant le contexte de l'un à l'autre.


## Création d'agents spécialisés


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Construire un flux de travail séquentiel

`WorkflowBuilder` vous permet de connecter des agents dans un graphe orienté. Ici, nous créons un pipeline simple en deux étapes : le **TravelPlanner** élabore l’itinéraire, puis le **TravelConcierge** le révise et l’améliore.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Ajout de plusieurs agents au flux de travail

L'un des plus grands avantages du modèle multi-agent est la facilité avec laquelle il peut être étendu. Ci-dessous, nous ajoutons un agent **BudgetReviewer** qui vérifie le plan par rapport au budget du voyageur, signale les éléments susceptibles de faire dépasser les coûts et suggère des alternatives économiques. Le flux de travail exécute désormais trois agents en séquence :

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Résumé

Dans cette leçon, vous avez appris à :

1. **Créer des agents spécialisés** — chacun avec un rôle précis (planification, conciergerie, revue du budget).
2. **Connecter les agents dans un flux de travail séquentiel** en utilisant `WorkflowBuilder` et `add_edge`.
3. **Diffuser la sortie** d’un pipeline multi-agent, en suivant quel agent parle.
4. **Étendre un flux de travail** en ajoutant de nouveaux agents à la chaîne sans modifier ceux existants.

Le modèle de conception multi-agent maintient chaque agent simple tout en produisant des résultats plus riches et plus minutieusement révisés que ce qu’un agent unique pourrait réaliser seul.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :  
Ce document a été traduit à l’aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçons d’assurer l’exactitude, veuillez noter que les traductions automatiques peuvent contenir des erreurs ou des imprécisions. Le document original dans sa langue d’origine doit être considéré comme la source faisant foi. Pour des informations critiques, il est recommandé de recourir à une traduction professionnelle effectuée par un humain. Nous déclinons toute responsabilité en cas de malentendus ou d’interprétations erronées résultant de l’utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
